# 06 Compute fund alpha and momentum

目标：计算费前/超额收益、t-36至t-1 Carhart暴露、project alpha及三个基金动量原始特征。

- 输入：05基金特征；FF月度与动量
- 输出：fund_alpha_momentum_raw.parquet及回归审计
- 主键：wficn+month
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：05
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import numpy as np, pandas as pd, polars as pl
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.rolling_regression import ols_with_intercept
from src.runtime_paths import output_path, upstream_path
from src.schema_utils import assert_unique

fund=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"05_aggregate_shareclasses_and_fund_features","data/processed/fund/fund_month_basic_features.parquet")).to_pandas()
def ff(path, names):
    x=pd.read_csv(path,dtype=str); x=x.rename(columns={x.columns[0]:"yyyymm"})
    for c in names: x[c]=pd.to_numeric(x[c].str.strip(),errors="coerce")/100
    x["month"]=pd.to_datetime(x["yyyymm"],format="%Y%m",errors="coerce")
    return x[["month"]+names]
factors=ff(PATHS["ff_monthly"],["mktrf","smb","hml","rf"]).merge(ff(PATHS["mom_monthly"],["mom"]),on="month",how="left")
fund["month"]=pd.to_datetime(fund["month"])
panel=fund.merge(factors,on="month",how="left",validate="many_to_one").sort_values(["wficn","month"])
panel["gross_return"]=panel["fund_return_net"]+panel["exp_ratio"]/12
panel["excess_return"]=panel["gross_return"]-panel["rf"]
factor_cols=["mktrf","smb","hml","mom"]; rows=[]
for wficn,g in panel.groupby("wficn",sort=False):
    g=g.sort_values("month").reset_index(drop=True)
    for i,row in g.iterrows():
        history=g.iloc[max(0,i-int(PARAMS["carhart_window_months"])):i]
        fit=ols_with_intercept(history["excess_return"].to_numpy(),history[factor_cols].to_numpy(),int(PARAMS["carhart_min_obs"]))
        beta=np.asarray(fit["beta"]); current=row[factor_cols].to_numpy(dtype=float)
        alpha=row["excess_return"]-beta@current if np.isfinite(row["excess_return"]) and np.all(np.isfinite(beta)) and np.all(np.isfinite(current)) else np.nan
        rows.append({"wficn":wficn,"month":row["month"],"excess_return":row["excess_return"],"alpha":alpha,"carhart_n_obs":fit["n_obs"],**{f"beta_{c}":beta[j] for j,c in enumerate(factor_cols)}})
result=(pd.DataFrame(rows).sort_values(["wficn","month"]) if rows else pd.DataFrame(columns=["wficn","month","excess_return","alpha","carhart_n_obs","beta_mktrf","beta_smb","beta_hml","beta_mom"]))
result["F_ST_Rev"]=result["alpha"]; result["F_r2_1"]=result["alpha"]
result["F_r12_2"]=result.groupby("wficn")["alpha"].transform(lambda s:s.shift(1).rolling(11,min_periods=int(PARAMS["fund_momentum_min_obs"])).mean())
assert result["F_ST_Rev"].equals(result["F_r2_1"])
out_frame=pl.from_pandas(result).with_columns(
    pl.col("month").cast(pl.Date,strict=False),
    pl.col("carhart_n_obs").cast(pl.Int64,strict=False),
    *[pl.col(c).cast(pl.Float64,strict=False) for c in ["excess_return","alpha","beta_mktrf","beta_smb","beta_hml","beta_mom","F_ST_Rev","F_r2_1","F_r12_2"]],
); assert_unique(out_frame,["wficn","month"],"project alpha panel")
assert "rret" not in out_frame.columns
out=output_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/processed/fund/fund_alpha_momentum_raw.parquet")
audit1=output_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/audit/06_carhart_insufficient_history.csv")
audit2=output_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/audit/06_carhart_beta_summary.csv")
write_parquet_atomic(out_frame,out)
write_csv_atomic(out_frame.filter(pl.col("carhart_n_obs")<int(PARAMS["carhart_min_obs"])),audit1)
write_csv_atomic(out_frame.select([c for c in out_frame.columns if c.startswith("beta_")]).describe(),audit2)
INPUTS_USED=["fund_month_basic_features.parquet","F-F_Research_Data_Factors.csv","F-F_Momentum_Factor.csv"]
OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2]]; INPUT_COUNTS={"fund_months":len(fund)}; OUTPUT_COUNTS={"alpha_months":out_frame.height}
PRIMARY_KEY=["wficn","month"]; DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=0; EXCEPTION_FILE=str(audit1.relative_to(ROOT))
STAGE_STATUS="passed"; STAGE_NOTES=["Regression window excludes current month; project alpha does not subtract the regression intercept."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "06_compute_fund_rret_and_momentum", stage="06", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
